# Phase 5i: Rank-2 Quantum Groups — Technical Notebook

**Phase:** 5i (Rank-2 Quantum Groups and SU(3)_k Fusion)  
**Date:** April 2026  
**Paper:** Paper 14 — *U_q(sl_3) and SU(3)_k Fusion Categories in Lean 4*

First rank-2 quantum group formalization in any proof assistant:
- **U_q(sl_3)**: 8 generators, A2 Cartan matrix, 21 Chevalley-Serre relations
- **Hopf algebra**: Coproduct/counit/antipode (4 sorry remaining)
- **SU(3)_1**: Z3 fusion ring, 3 simple objects, charge conjugation
- **SU(3)_2**: 6 anyons, Fibonacci subcategory (tau x tau = 1 + tau)
- **Q(zeta_3)**: Cyclotomic field for S-matrix entries
- **PolyQuotQ**: Polynomial quotient ring infrastructure

---

In [ ]:
import numpy as np
from src.core.formulas import (
    su2k_fusion_rule, su2k_quantum_dim, su2k_s_matrix_entry,
    q_integer, q_dg_coefficient,
    uqsl2_coproduct, uqsl2_counit, uqsl2_antipode,
)
from src.core.visualizations import COLORS

## 1. U_q(sl_3): Generators and Chevalley-Serre Relations

U_q(sl_3) has 8 generators: E1, E2, F1, F2, K1, K1inv, K2, K2inv.  
The A2 Cartan matrix is:
```
A = [[ 2, -1],
     [-1,  2]]
```
This gives 21 Chevalley-Serre relations in the quotient algebra.

In [ ]:
# A2 Cartan matrix
A2 = np.array([[2, -1], [-1, 2]])
print("A2 Cartan matrix:")
print(A2)
print()

# Generators of U_q(sl_3)
generators = ['E1', 'E2', 'F1', 'F2', 'K1', 'K1inv', 'K2', 'K2inv']
print(f"Generators ({len(generators)} total): {', '.join(generators)}")
print()

# The 21 Chevalley-Serre relations
relations = [
    'K1 * K1inv = 1', 'K1inv * K1 = 1',
    'K2 * K2inv = 1', 'K2inv * K2 = 1',
    'K1 * K2 = K2 * K1',
    'K1 * E1 = q^2 * E1 * K1', 'K1 * E2 = q^(-1) * E2 * K1',
    'K2 * E1 = q^(-1) * E1 * K2', 'K2 * E2 = q^2 * E2 * K2',
    'K1 * F1 = q^(-2) * F1 * K1', 'K1 * F2 = q * F2 * K1',
    'K2 * F1 = q * F1 * K2', 'K2 * F2 = q^(-2) * F2 * K2',
    'E1*F1 - F1*E1 = (K1 - K1inv)/(q - qinv)',
    'E2*F2 - F2*E2 = (K2 - K2inv)/(q - qinv)',
    'E1*F2 = F2*E1', 'E2*F1 = F1*E2',
    'E1^2*E2 - (q+qinv)*E1*E2*E1 + E2*E1^2 = 0 (Serre)',
    'E2^2*E1 - (q+qinv)*E2*E1*E2 + E1*E2^2 = 0 (Serre)',
    'F1^2*F2 - (q+qinv)*F1*F2*F1 + F2*F1^2 = 0 (Serre)',
    'F2^2*F1 - (q+qinv)*F2*F1*F2 + F1*F2^2 = 0 (Serre)',
]
print(f"Chevalley-Serre relations: {len(relations)}")
for i, rel in enumerate(relations):
    print(f"  R{i+1:2d}: {rel}")

print()
print("Lean: Uqsl3.lean — 21 theorems (all Chevalley-Serre relations verified, 0 sorry)")

## 2. Hopf Algebra Structure

The Hopf algebra structure on U_q(sl_3) consists of:
- **Coproduct** Delta: U_q -> U_q tensor U_q
- **Counit** epsilon: U_q -> k
- **Antipode** S: U_q -> U_q (algebra anti-homomorphism)

The 4 sorry are all relation-respect proofs (showing the maps descend
from the free algebra to the quotient).

In [ ]:
# Hopf algebra maps on generators
print("Coproduct Delta (on generators):")
print("  Delta(E_i) = E_i (x) K_i + 1 (x) E_i")
print("  Delta(F_i) = F_i (x) 1 + K_i^{-1} (x) F_i")
print("  Delta(K_i) = K_i (x) K_i")
print("  Delta(K_i^{-1}) = K_i^{-1} (x) K_i^{-1}")
print()

print("Counit epsilon:")
print("  epsilon(E_i) = epsilon(F_i) = 0")
print("  epsilon(K_i) = epsilon(K_i^{-1}) = 1")
print()

print("Antipode S (anti-homomorphism):")
print("  S(K_i) = K_i^{-1},  S(K_i^{-1}) = K_i")
print("  S(E_i) = -E_i K_i^{-1}")
print("  S(F_i) = -K_i F_i")
print()

print("Key identity: S^2(x) = Ad(K1*K2)(x) = K1*K2 * x * K2inv*K1inv")
print()

# Compare with sl_2 case (which is fully proved)
print("Comparison with U_q(sl_2) (fully proved):")
for gen_name in ['E', 'F', 'K']:
    counit = uqsl2_counit(gen_name)
    print(f"  epsilon_sl2({gen_name}) = {counit}")

print()
print("Lean: Uqsl3Hopf.lean — 4 sorry (all relation-respect for ring quotient descent)")
print("  3 sorry: comul/counit/antipode respect Chevalley relations")
print("  1 sorry: S^2 = Ad(K1*K2) identity")

## 3. SU(3)_1: Z3 Fusion Ring

SU(3) at level k=1 has (k+1)(k+2)/2 = 3 simple objects:
vacuum (0,0), fundamental (1,0), conjugate (0,1).
Fusion is isomorphic to Z3 addition.

In [ ]:
# SU(3)_1: Z3 fusion ring
labels_k1 = ['vac', 'f', 'f-bar']
# Fusion table: Z3 addition
# vac(x)x = x, f(x)f = f-bar, f(x)f-bar = vac, f-bar(x)f-bar = f
z3_table = [
    [0, 1, 2],  # vac (x) {vac, f, f-bar}
    [1, 2, 0],  # f (x) {vac, f, f-bar}
    [2, 0, 1],  # f-bar (x) {vac, f, f-bar}
]

print("SU(3)_1 fusion table (Z3 group ring):")
header = f"{'':8s}" + ''.join(f"{l:8s}" for l in labels_k1)
print(header)
for i, row in enumerate(z3_table):
    entries = ''.join(f"{labels_k1[j]:8s}" for j in row)
    print(f"{labels_k1[i]:8s}{entries}")

print()
print("Properties:")
print("  - 3 simple objects = (k+1)(k+2)/2 for k=1")
print("  - All multiplicities are 0 or 1 (multiplicity-free)")
print("  - Charge conjugation: f-bar = conjugate of f")
print("  - Group-like: fusion ring = Z_3")
print()
print("Lean: SU3kFusion.lean — su3k1_object_count, su3k1_is_z3")

## 4. SU(3)_2: 6 Anyons and Fibonacci Subcategory

SU(3) at level k=2 has (k+1)(k+2)/2 = 6 simple objects.  
Crucially, it contains a **Fibonacci subcategory**: the object tau
satisfies tau x tau = 1 + tau, making it universal for quantum computation.

In [ ]:
# SU(3)_2: 6 anyons
# Simple objects: (0,0)=1, (1,0)=f, (0,1)=f-bar, (2,0)=s, (0,2)=s-bar, (1,1)=tau
labels_k2 = ['1', 'f', 'f-bar', 's', 's-bar', 'tau']
n_objects = 6

print(f"SU(3)_2: {n_objects} simple objects = (k+1)(k+2)/2 for k=2")
print()

# Dominant weights
weights = [(0,0), (1,0), (0,1), (2,0), (0,2), (1,1)]
print("Dominant weights (lambda_1, lambda_2) with lambda_1+lambda_2 <= k=2:")
for label, (l1, l2) in zip(labels_k2, weights):
    print(f"  {label:6s}: ({l1},{l2})")

print()

# Key fusion rules
print("Key fusion rules:")
print("  f (x) f = f-bar + s       (2 channels)")
print("  f (x) f-bar = 1 + tau     (2 channels)")
print("  tau (x) tau = 1 + tau     (Fibonacci!)")
print("  s (x) s-bar = 1           (simple)")
print()

# Fibonacci subcategory
phi = (1 + np.sqrt(5)) / 2
print(f"Fibonacci subcategory {{1, tau}}:")
print(f"  tau (x) tau = 1 + tau")
print(f"  Quantum dimension d_tau = phi = {phi:.6f}")
print(f"  This subcategory is UNIVERSAL for quantum computation.")
print()
print("Lean: SU3kFusion.lean — su3k2_object_count, su3k2_fibonacci_subcategory")

## 5. Q(zeta_3) Cyclotomic Field for S-matrix

The S-matrix of SU(3)_1 lives in Q(zeta_3), the third cyclotomic field.  
zeta_3 = exp(2*pi*i/3) = (-1 + i*sqrt(3))/2.

The polynomial quotient ring Q[x]/(x^2 + x + 1) provides the exact arithmetic.

In [ ]:
# Q(zeta_3) cyclotomic field
zeta3 = np.exp(2j * np.pi / 3)
print(f"zeta_3 = exp(2*pi*i/3) = {zeta3:.6f}")
print(f"Minimal polynomial: x^2 + x + 1 = 0")
print(f"Check: zeta_3^2 + zeta_3 + 1 = {zeta3**2 + zeta3 + 1:.2e}")
print()

# SU(3)_1 S-matrix (3x3, over Q(zeta_3))
# S_{ab} = (1/sqrt(3)) * zeta_3^{ab} for a,b in {0,1,2}
S_k1 = np.array([[zeta3**(a*b) for b in range(3)] for a in range(3)]) / np.sqrt(3)
print("SU(3)_1 S-matrix (normalized):")
for i in range(3):
    row = '  '.join(f"{S_k1[i,j]:.4f}" for j in range(3))
    print(f"  [{row}]")

# Unitarity check
I_check = S_k1 @ S_k1.conj().T
max_dev = np.max(np.abs(I_check - np.eye(3)))
print(f"\nS * S^dagger = I max deviation: {max_dev:.2e}")
print(f"det(S) = {np.linalg.det(S_k1):.4f}")
print()
print("Lean: PolyQuotQ.lean — 15 theorems (Q[x]/(x^2+x+1) ring infrastructure)")
print("Lean: SU3kFusion.lean — S-matrix unitarity for k=1, k=2")

## 6. Comparison: SU(2)_k vs SU(3)_k

The rank-2 case is significantly more complex than rank-1.

In [ ]:
# Object count comparison
print("Object count comparison:")
print(f"{'Level k':>8s} {'SU(2)_k':>10s} {'SU(3)_k':>10s} {'Ratio':>8s}")
for k in range(1, 7):
    su2_count = k + 1
    su3_count = (k + 1) * (k + 2) // 2
    print(f"{k:>8d} {su2_count:>10d} {su3_count:>10d} {su3_count/su2_count:>8.1f}x")

print()

# SU(2)_k quantum dimensions for comparison
for k in [1, 2, 3]:
    n = k + 1
    dims = [round(su2k_quantum_dim(k, j), 4) for j in range(n)]
    print(f"SU(2)_{k} quantum dims: {dims}")

print()
print("Key distinction: SU(3)_k has rank-2 weight lattice,")
print("requiring A2 Cartan matrix and 2-index Serre relations.")
print("This is the first rank-2 quantum group in any proof assistant.")

## 7. Provenance

| Lean Module | Theorems | Sorry | Key Result |
|-------------|----------|-------|------------|
| Uqsl3.lean | 21 | 0 | 8 generators, A2 Cartan, 21 Chevalley-Serre relations |
| Uqsl3Hopf.lean | 4 | 4 | Coproduct/counit/antipode (relation-respect sorry) |
| SU3kFusion.lean | 99 | 0 | Z3 fusion (k=1), 6 anyons + Fibonacci (k=2) |
| PolyQuotQ.lean | 15 | 0 | Q(zeta_3) cyclotomic field infrastructure |

Total: 139 theorems, 4 sorry (all in Hopf relation-respect proofs).  
First rank-2 quantum group formalization in any proof assistant.